# Correcting a segmentation in the Cellpose GUI (one annotation round)

Exports one segmentation of the store written by notebook 01 for the Cellpose GUI, opens it, and records the round.
Each image of each round gets its own folder, `$MACROSEG_ANNOTATIONS/round_<NN>/<name>/` (see `macroseg.annotation`):

- `<name>.tif`: three image channels, shown as red, green and blue (the GUI keeps only the first three);
- `<name>_seg.npy`: the masks, loaded by the GUI; Cmd + S saves the corrections into it;
- `start_masks.tif`: the masks as exported, to count the changes;
- `annotation.json`: round, role (`train` / `test`), where image and masks come from, and after section 3
  what was changed and how well the starting masks scored.

In the GUI: right-click and draw to add a cell, Cmd + click to delete one, Cmd + S to save.
Review the **whole** image: untouched cells count as correct.

In [9]:
from pathlib import Path

from spatialdata import read_zarr

from macroseg import paths
from macroseg.annotation import export_for_gui, finish, open_gui
from macroseg.config import load_config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_test_crop.yaml")

ROUND = 1                                   # annotation round
ROLE = "train"                              # "train" or "test" (a test image is never used for training)
SEGMENTATION = "membrane_dapi"              # starting masks: a labels element of the store
CHANNELS = ["F480", "NaK_ATPase", "DAPI"]   # red, green, blue in the GUI
OVERWRITE = False                           # True: export again and discard the corrections saved in the GUI

## 1. Export image and masks
If the folder already holds `<name>_seg.npy`, nothing is exported (it may hold your corrections) unless `OVERWRITE = True`.

In [10]:
sdata = read_zarr(cfg.zarr)
folder = export_for_gui(
    sdata, cfg, SEGMENTATION, CHANNELS, paths.path("MACROSEG_ANNOTATIONS"), ROUND, role=ROLE, overwrite=OVERWRITE
)
folder

no parent found for <ome_zarr.reader.Label object at 0x11d1f6350>: None
no parent found for <ome_zarr.reader.Label object at 0x11d1f5470>: None
no parent found for <ome_zarr.reader.Label object at 0x11d38c1f0>: None
no parent found for <ome_zarr.reader.Label object at 0x12b74abd0>: None
no parent found for <ome_zarr.reader.Label object at 0x12b74ab10>: None
no parent found for <ome_zarr.reader.Label object at 0x11d1aec50>: None


PosixPath('/Users/massimoz/Documents/macroseg/data/annotations/round_01/exp201_test_crop')

## 2. Open the Cellpose GUI
A separate process: the notebook stays free, and closing the GUI does not stop the kernel.

In [11]:
open_gui(folder);

## 3. After annotating: record the changes
Save in the GUI first (Cmd + S). `complete=False` if the image was not reviewed entirely.
Can be run again after further corrections.

In [ ]:
record = finish(folder, complete=True)
record["counts"], record["start_score"]

({'cells_start': 1587,
  'cells_final': 1550,
  'unchanged': 1533,
  'removed': 54,
  'drawn': 17,
  'changed_%': 3.4},
 {'AP@0.5': 0.9631,
  'precision@0.5': 0.9698,
  'recall@0.5': 0.9929,
  'AP@0.75': 0.9557,
  'precision@0.75': 0.966,
  'recall@0.75': 0.989})

[GUI INFO] : WRITING LOG OUTPUT TO /Users/massimoz/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	darwin 
python version: 	3.13.16 
torch version:  	2.14.1
2026-10-27 10:57:54,906 [io INFO] WRITING LOG OUTPUT TO /Users/massimoz/.cellpose/run.log
2026-10-27 10:57:54,906 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	darwin 
python version: 	3.13.16 
torch version:  	2.14.1
2026-10-27 10:57:55,149 [core INFO] ** TORCH MPS version installed and working. **
2026-10-27 10:57:55,221 [io INFO]  : image shape: (2048, 2048, 3)
(1, 2048, 2048, 3)
{'lowhigh': None, 'percentile': [1.0, 99.0], 'normalize': True, 'norm3D': True, 'sharpen_radius': 0.0, 'smooth_radius': 0.0, 'tile_norm_blocksize': 0.0, 'tile_norm_smooth3D': 1.0, 'invert': False}
(1, 2048, 2048, 3)
2026-10-27 10:57:55,471 [io INFO] 1550 masks found
2026-10-27 10:57:55,509 [io INFO] creating cellcolors and drawing masks
2026-10-27 10:57:55,600 [io INFO] loaded in previous changes
2026-10-27 10:58:00,476 [io INF